In [ ]:
# ── Path setup ───────────────────────────────────────────────────────────
# All paths are derived from BASE_DIR. Override CAPSTONE_BASE_DIR if your
# trained model and data live somewhere else.
import os, sys

BASE_DIR   = os.environ.get("CAPSTONE_BASE_DIR", os.getcwd())
MODEL_DIR  = os.path.join(BASE_DIR, "trained_model")
DATA_DIR   = os.path.join(BASE_DIR, "data")
DEPLOY_CSV = os.path.join(DATA_DIR, "test_data.csv")  # produced by train_model

sys.path.insert(0, BASE_DIR)
print(f"BASE_DIR = {BASE_DIR}")
print(f"  Model expected at: {os.path.join(MODEL_DIR, 'model_weights.pt')}")
print(f"  Deploy CSV: {DEPLOY_CSV}")

## Install dependencies

In [ ]:
!pip install -q scikit-learn psycopg2-binary torch

## Imports

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, precision_recall_curve
import pickle, json, uuid, importlib
import sys

sys.path.insert(0, BASE_DIR)

# The model architecture must match what train_model produced; reload
# the module so any local edits are picked up.
from model_components import (
    LongRangeFraudTransformer,
    engineer_behavioral_features,
    PaySimSequenceDataset,
    FEATURE_COLS,
)

# db_scoring is optional — if the database can't be reached, deployment
# falls back to using only the Transformer score.
try:
    import db_scoring
    importlib.reload(db_scoring)
    DB_AVAILABLE = True
    print("✓ db_scoring loaded")
except ImportError:
    DB_AVAILABLE = False
    print("[WARNING] db_scoring not available - SQL scoring disabled")

## Helper functions

In [ ]:
def load_model(model_dir=MODEL_DIR):
    """Reconstruct the model from the artefacts saved by train_model.

    Reads config.json, instantiates the same architecture, and loads the
    weights, scaler, and label encoder so features look identical to
    training time.
    """
    print(f"Loading model from {model_dir}/...")

    with open(os.path.join(model_dir, "config.json")) as f:
        config = json.load(f)

    # The .get() defaults exist so old checkpoints (saved before the
    # type-embedding refactor) still load with sensible values.
    model = LongRangeFraudTransformer(
        num_features  = config["num_features"],
        context_len   = config["context_length"],
        num_type_cats = config.get("num_type_cats", 2),
        type_emb_dim  = config.get("type_emb_dim",  8),
        type_col_idx  = config.get("type_col_idx",  5),
    )
    model.load_state_dict(torch.load(
        os.path.join(model_dir, "model_weights.pt"),
        map_location="cpu", weights_only=False))
    model.eval()

    with open(os.path.join(model_dir, "scaler.pkl"), "rb") as f:
        scaler = pickle.load(f)

    with open(os.path.join(model_dir, "label_encoder.pkl"), "rb") as f:
        label_encoder = pickle.load(f)

    print(f"✓ Model loaded  |  features={config['num_features']}"
          f"  context_len={config['context_length']}"
          f"  type_emb_dim={config.get('type_emb_dim', 'N/A (old checkpoint)')}")
    return model, scaler, label_encoder, config


def prepare_deployment_data(filepath, label_encoder, context_len):
    """Read the deploy CSV, engineer features, and (optionally) push to DB."""
    print(f"\nLoading deployment data from {filepath}...")
    df = pd.read_csv(filepath)
    df = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].copy()

    df["type_enc"] = label_encoder.transform(df["type"])
    df = df.sort_values(by=["nameOrig", "step"]).reset_index(drop=True)
    df = engineer_behavioral_features(df)
    df = df.sort_values("step").reset_index(drop=True)

    print(f"Deployment data: {len(df)} transactions")

    # If the DB is up, insert the rows so SQL scoring can find them.
    # Otherwise just attach random UUIDs so the rest of the pipeline
    # has a per-row identifier to work with.
    if DB_AVAILABLE:
        print("\n>>> Inserting new data into PostgreSQL...")
        try:
            uuids = db_scoring.insert_transactions(df)
            df["transaction_id"] = uuids
            print(f"    ✓ {len(uuids)} transactions inserted")
        except Exception as e:
            print(f"    ✗ Database insert failed: {e}")
            df["transaction_id"] = [str(uuid.uuid4()) for _ in range(len(df))]
    else:
        df["transaction_id"] = [str(uuid.uuid4()) for _ in range(len(df))]

    return df


def get_sql_scores(df, context_len):
    """Pull SQL anomaly scores aligned to the global sliding-window sequences.

    For each window the Transformer produces, we want the SQL score for
    the *last* transaction in that window — that's the row being predicted.
    Returns zeros if the database is unavailable so fusion can still run.
    """
    if not DB_AVAILABLE:
        print("  (Database unavailable - skipping SQL scoring)")
        return np.zeros(max(0, len(df) - context_len))

    try:
        print(">>> Computing SQL scores via PostgreSQL...")
        uuids     = df["transaction_id"].tolist()
        scores_df = db_scoring.score_batch(uuids)
        score_map = dict(zip(scores_df["transaction_id"], scores_df["sql_anomaly_score"]))

        window     = context_len + 1
        sql_scores = [
            score_map.get(uuids[i + context_len], 0.0)
            for i in range(0, len(uuids) - window + 1)
        ]
        return np.array(sql_scores)

    except Exception as e:
        print(f"  ✗ SQL scoring failed: {e}")
        return np.zeros(max(0, len(df) - context_len))

## Deployment — run all cells above, then this one

In [ ]:
def deploy(data_path=DEPLOY_CSV, model_dir=MODEL_DIR):
    """End-to-end deployment pipeline.

    Steps:
      1. Load the trained Transformer and its preprocessing artefacts.
      2. Read and feature-engineer the deployment data.
      3. Run inference and squash logits with sigmoid.
      4. Open the DB pool *after* inference — keeps idle connections
         from timing out during long CPU-only inference runs.
      5. Fuse Transformer + SQL scores (0.8 / 0.2 by default).
      6. Pick a threshold (best F1) and flag the high-risk rows.
    """
    # 1. Load model and artefacts.
    model, scaler, le, config = load_model(model_dir)
    context_len  = config["context_length"]
    feature_cols = config["feature_cols"]

    # 2. Prepare deployment data (also pushes to DB if available).
    df = prepare_deployment_data(data_path, le, context_len)

    # 3. Build the sliding-window inference dataset (no masking, no shuffle).
    dataset = PaySimSequenceDataset(
        df, scaler=scaler, context_len=context_len,
        train=False, feature_cols=feature_cols, stride=1,
    )
    print(f"Inference dataset: {len(dataset)} sequences")

    # 4. Transformer inference loop. Uses GPU if available.
    print("\n>>> Running Transformer inference...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"    Device: {device}")
    model  = model.to(device)

    recon_errors   = []
    anomaly_scores = []

    loader = DataLoader(dataset, batch_size=64, shuffle=False)
    with torch.no_grad():
        for batch in loader:
            out = model(
                past_values        = batch["past_values"].to(device),
                past_time_features = batch["past_time_features"].to(device),
                future_values      = batch["future_values"].to(device),
            )
            # Reconstruction error — bigger means the next txn was unexpected.
            mse = F.mse_loss(
                out["reconstruction"],
                batch["future_values"].to(device),
                reduction="none",
            ).mean(dim=(1, 2))
            recon_errors.extend(mse.cpu().numpy())

            # Direct fraud probability from the anomaly head.
            probs = torch.sigmoid(out["anomaly_score"]).squeeze(-1)
            anomaly_scores.extend(probs.cpu().numpy())

    # Combine the two Transformer signals into one score.
    # Reconstruction error gets normalised to [0, 1] so the weighted sum
    # is meaningful regardless of how big the raw MSEs are.
    t_scores = np.array(recon_errors)
    t_norm   = (t_scores - t_scores.min()) / (t_scores.max() - t_scores.min() + 1e-9)
    transformer_scores = 0.7 * t_norm + 0.3 * np.array(anomaly_scores)

    # 5. Open the DB pool *now*, after inference is done. Doing it earlier
    # risks the connection going idle and being killed mid-run.
    if DB_AVAILABLE:
        try:
            db_scoring.init_pool()
            print("✓ DB pool initialised")
        except Exception as e:
            print(f"[WARNING] DB pool init failed: {e}")

    # 6. Pull SQL scores aligned to the same sliding windows.
    sql_scores = get_sql_scores(df, context_len)

    # 7. Score fusion. The 0.8 / 0.2 weighting was the report's sweet spot.
    n     = min(len(transformer_scores), len(sql_scores))
    fused = 0.8 * transformer_scores[:n] + 0.2 * sql_scores[:n]

    # 8. Choose a threshold (best F1 on the labels) and report metrics.
    predictions = None
    threshold   = 0.5

    if "isFraud" in df.columns:
        true_labels = [dataset[i]["labels"].item() for i in range(n)]

        if len(set(true_labels)) < 2:
            print("Warning: only one class in labels — using threshold=0.5")
        else:
            precisions, recalls, thresholds = precision_recall_curve(true_labels, fused)
            f1s       = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-9)
            threshold = float(thresholds[np.argmax(f1s)])
            print(f"Best F1 threshold: {threshold:.4f}")

        predictions = (fused > threshold).astype(int)
        print(f"\n=== Results ===")
        print(f"Threshold: {threshold:.4f}")
        print(classification_report(true_labels, predictions, target_names=["Normal", "Fraud"]))

    # 9. Persist the high-risk transactions in the DB (best-effort).
    if DB_AVAILABLE:
        try:
            all_uuids = df["transaction_id"].tolist()
            window    = context_len + 1
            txn_ids   = [all_uuids[i + context_len] for i in range(0, len(all_uuids) - window + 1)]
            n_flagged = db_scoring.flag_high_risk(txn_ids[:n], fused, threshold)
            print(f"\n✓ {n_flagged} high-risk transactions flagged in database")
        except Exception as e:
            print(f"Could not flag transactions: {e}")
        finally:
            db_scoring.close_pool()  # close *after* flagging — order matters

    # 10. Save scores to a CSV so they can be inspected outside the notebook.
    if predictions is None:
        predictions = (fused > threshold).astype(int)

    results_df = pd.DataFrame({
        "fused_score":       fused,
        "prediction":        predictions[:n],
        "transformer_score": transformer_scores[:n],
        "sql_score":         sql_scores[:n],
    })
    out_path = os.path.join(BASE_DIR, "deployment_results.csv")
    results_df.to_csv(out_path, index=False)
    print(f"\n✓ Results saved to {out_path}")

    return results_df


# ── Run deployment ──────────────────────────────────────────────────────
results = deploy(data_path=DEPLOY_CSV, model_dir=MODEL_DIR)
results.head()